# 12. Final XGBoost Recommendation Revalidation

## Objective

The purpose of this notebook is to revalidate the personalized recommendation engine after the final predictive model was changed from Random Forest to XGBoost.

The final frozen XGBoost model and frozen SHAP thresholds are used without further model tuning.

The recommendation logic combines construct-level scores and local SHAP contributions to identify strengths, monitoring areas, and development priorities.

This notebook evaluates the technical consistency and behaviour of the final recommendation engine. It is not an additional predictive performance evaluation because predictive model validation was completed using nested cross-validation in the previous stage.

Formal domain-expert validation of the recommendation content remains pending.

In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import shap

d:\internship-employability-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent


DATA_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "descriptive_dataset.csv"
)

MODEL_FILE = (
    PROJECT_DIR
    / "models"
    / "final_employability_model.joblib"
)

CONFIG_FILE = (
    PROJECT_DIR
    / "models"
    / "recommendation_system_config_frozen.json"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "reports"
    / "final_recommendation_validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Dataset:", DATA_FILE)
print("Model:", MODEL_FILE)
print("Config:", CONFIG_FILE)

Dataset: d:\internship-employability-project\data\processed\descriptive_dataset.csv
Model: d:\internship-employability-project\models\final_employability_model.joblib
Config: d:\internship-employability-project\models\recommendation_system_config_frozen.json


In [3]:
final_model = joblib.load(
    MODEL_FILE
)

print(
    "Loaded model:",
    type(final_model).__name__
)

Loaded model: XGBRegressor


In [4]:
with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as file:

    recommendation_config = json.load(
        file
    )

In [5]:
print(
    "Config model:",
    recommendation_config[
        "model_name"
    ]
)

print(
    "Small SHAP threshold:",
    recommendation_config[
        "small_shap_threshold"
    ]
)

print(
    "Large SHAP threshold:",
    recommendation_config[
        "large_shap_threshold"
    ]
)

print(
    "Validation status:",
    recommendation_config[
        "validation_status"
    ]
)

Config model: XGBoost
Small SHAP threshold: 0.02683858573436737
Large SHAP threshold: 0.07688135653734207
Validation status: Technical prototype validated; expert content validation pending


Verify frozen model/config consistency

In [6]:
assert (
    recommendation_config[
        "model_name"
    ]
    == "XGBoost"
), "Frozen configuration is not linked to XGBoost."

print(
    "Frozen model/config consistency check passed."
)

Frozen model/config consistency check passed.


Load dataset

In [7]:
df = pd.read_csv(
    DATA_FILE
)

feature_names = [
    "Clear_Objectives",
    "School_Support",
    "Workplace_Environment",
    "Career_Development"
]

target_name = (
    "Employability_Score"
)

X = df[
    feature_names
].copy()

y = df[
    target_name
].copy()

print(
    "Evaluation records:",
    len(X)
)

print(
    "Features:",
    feature_names
)

Evaluation records: 1000
Features: ['Clear_Objectives', 'School_Support', 'Workplace_Environment', 'Career_Development']


Load frozen SHAP thresholds

In [8]:
small_effect_threshold = float(
    recommendation_config[
        "small_shap_threshold"
    ]
)

large_effect_threshold = float(
    recommendation_config[
        "large_shap_threshold"
    ]
)

print(
    "Frozen small threshold:",
    small_effect_threshold
)

print(
    "Frozen large threshold:",
    large_effect_threshold
)

Frozen small threshold: 0.02683858573436737
Frozen large threshold: 0.07688135653734207


Load recommendation mapping

In [9]:
recommendation_map = (
    recommendation_config[
        "recommendations"
    ]
)

print(
    recommendation_map.keys()
)

dict_keys(['Clear_Objectives', 'School_Support', 'Workplace_Environment', 'Career_Development'])


Create final XGBoost SHAP explanations

In [10]:
final_explainer = shap.TreeExplainer(
    final_model
)

final_shap_values = final_explainer(
    X
)

print(
    "SHAP shape:",
    final_shap_values.values.shape
)

SHAP shape: (1000, 4)


Construct-score classification

In [11]:
def classify_construct_score(
    score: float
) -> str:

    if score < 3.0:
        return "Low"

    elif score < 4.0:
        return "Moderate"

    return "High"

SHAP effect classification

In [12]:
def classify_shap_effect(
    shap_value: float,
    small_threshold: float,
    large_threshold: float
) -> str:

    magnitude = abs(
        shap_value
    )

    if magnitude < small_threshold:

        magnitude_label = "Small"

    elif magnitude < large_threshold:

        magnitude_label = "Moderate"

    else:

        magnitude_label = "Large"

    if shap_value >= 0:

        direction = "Positive"

    else:

        direction = "Negative"

    return (
        f"{magnitude_label} "
        f"{direction}"
    )

Final hybrid priority logic

In [13]:
def determine_priority(
    score_level: str,
    shap_value: float,
    small_threshold: float,
    large_threshold: float
) -> str:

    magnitude = abs(
        shap_value
    )

    # High development need
    if (
        score_level == "Low"
        and shap_value < 0
        and magnitude >= large_threshold
    ):
        return "High Priority"

    # Low score + negative effect
    if (
        score_level == "Low"
        and shap_value < 0
    ):
        return "Priority"

    # Moderate score + meaningful negative effect
    if (
        score_level == "Moderate"
        and shap_value < 0
        and magnitude >= small_threshold
    ):
        return "Moderate Priority"

    # High score + positive model contribution
    if (
        score_level == "High"
        and shap_value > 0
    ):
        return "Strength"

    # Moderate score + positive contribution
    if (
        score_level == "Moderate"
        and shap_value > 0
    ):
        return "Developing Strength"

    return "Monitor"

In [14]:
priority_order = {

    "High Priority": 1,

    "Priority": 2,

    "Moderate Priority": 3,

    "Monitor": 4,

    "Developing Strength": 5,

    "Strength": 6
}

Analyse one student's constructs

In [15]:
def analyse_student_constructs(
    student_position: int
):

    student_input = X.iloc[
        [student_position]
    ]

    student_shap = (
        final_shap_values[
            student_position
        ]
    )

    analysis = pd.DataFrame({

        "Construct":
            feature_names,

        "Construct Score":
            student_input.iloc[
                0
            ].values,

        "SHAP Contribution":
            student_shap.values
    })

    analysis[
        "Score Level"
    ] = (
        analysis[
            "Construct Score"
        ]
        .apply(
            classify_construct_score
        )
    )

    analysis[
        "SHAP Effect"
    ] = (
        analysis[
            "SHAP Contribution"
        ]
        .apply(
            lambda value:
                classify_shap_effect(
                    value,
                    small_effect_threshold,
                    large_effect_threshold
                )
        )
    )

    analysis[
        "Priority"
    ] = (
        analysis.apply(
            lambda row:
                determine_priority(
                    row[
                        "Score Level"
                    ],
                    row[
                        "SHAP Contribution"
                    ],
                    small_effect_threshold,
                    large_effect_threshold
                ),
            axis=1
        )
    )

    analysis[
        "Priority Rank"
    ] = (
        analysis[
            "Priority"
        ]
        .map(
            priority_order
        )
    )

    return (
        analysis.sort_values(
            [
                "Priority Rank",
                "SHAP Contribution"
            ],
            ascending=[
                True,
                True
            ]
        )
    )

Generate targeted recommendations

In [16]:
def generate_recommendations(
    analysis,
    maximum_constructs=2
):

    development_labels = {
        "High Priority",
        "Priority",
        "Moderate Priority"
    }

    priority_rows = (
        analysis[
            analysis[
                "Priority"
            ].isin(
                development_labels
            )
        ]
        .head(
            maximum_constructs
        )
    )

    recommendations = []

    for _, row in (
        priority_rows.iterrows()
    ):

        construct = (
            row[
                "Construct"
            ]
        )

        recommendations.append({

            "Construct":
                construct,

            "Construct Score":
                round(
                    float(
                        row[
                            "Construct Score"
                        ]
                    ),
                    2
                ),

            "SHAP Contribution":
                round(
                    float(
                        row[
                            "SHAP Contribution"
                        ]
                    ),
                    4
                ),

            "Priority":
                row[
                    "Priority"
                ],

            "Reason":
                (
                    f"{row['Score Level']} construct score "
                    f"with {row['SHAP Effect']} model contribution"
                ),

            "Actions":
                recommendation_map[
                    construct
                ]
        })

    return recommendations

Maintenance feedback

In [17]:
def generate_maintenance_message(
    strengths
):

    if strengths:

        strength_names = [
            item[
                "Construct"
            ].replace(
                "_",
                " "
            )
            for item in strengths
        ]

        strength_text = ", ".join(
            strength_names
        )

        return (
            "No priority development area was identified. "
            f"Maintain your current strengths in {strength_text} "
            "and continue seeking opportunities for professional development."
        )

    return (
        "No priority development area was identified. "
        "Continue maintaining your current internship performance "
        "and seek further professional development opportunities."
    )

Final report generator

In [18]:
def create_final_recommendation_report(
    student_position: int,
    include_actual=True
):

    if not (
        0 <= student_position < len(X)
    ):

        raise IndexError(
            "Student position is outside the dataset."
        )

    student_input = X.iloc[
        [student_position]
    ]

    prediction = float(
        final_model.predict(
            student_input
        )[0]
    )

    analysis = (
        analyse_student_constructs(
            student_position
        )
    )

    recommendations = (
        generate_recommendations(
            analysis,
            maximum_constructs=2
        )
    )

    strengths_df = (
        analysis[
            analysis[
                "Priority"
            ].isin(
                [
                    "Strength",
                    "Developing Strength"
                ]
            )
        ]
        .sort_values(
            "SHAP Contribution",
            ascending=False
        )
    )

    strengths = (
        strengths_df[
            [
                "Construct",
                "Construct Score",
                "SHAP Contribution"
            ]
        ]
        .to_dict(
            "records"
        )
    )

    if recommendations:

        maintenance_message = None

    else:

        maintenance_message = (
            generate_maintenance_message(
                strengths
            )
        )

    report = {

        "Student Position":
            int(
                student_position
            ),

        "Original Row Index":
            int(
                X.index[
                    student_position
                ]
            ),

        "Predicted Employability Score":
            round(
                prediction,
                3
            ),

        "Strengths":
            strengths,

        "Development Recommendations":
            recommendations,

        "Maintenance Message":
            maintenance_message,

        "Full Construct Analysis":
            analysis.drop(
                columns=[
                    "Priority Rank"
                ]
            ).to_dict(
                "records"
            )
    }

    # Research inspection only
    if include_actual:

        actual = float(
            y.iloc[
                student_position
            ]
        )

        report[
            "Actual Employability Score"
        ] = round(
            actual,
            3
        )

    return report

Generate predictions for representative cases

In [19]:
all_predictions = (
    final_model.predict(
        X
    )
)

In [20]:
sorted_positions = np.argsort(
    all_predictions
)

low_positions = (
    sorted_positions[
        :3
    ].tolist()
)

high_positions = (
    sorted_positions[
        -3:
    ][::-1].tolist()
)

median_prediction = np.median(
    all_predictions
)

middle_positions = (
    np.argsort(
        np.abs(
            all_predictions
            - median_prediction
        )
    )[:3].tolist()
)

print(
    "Low positions:",
    low_positions
)

print(
    "Middle positions:",
    middle_positions
)

print(
    "High positions:",
    high_positions
)

Low positions: [197, 213, 223]
Middle positions: [989, 988, 985]
High positions: [62, 936, 64]


In [21]:
low_report = (
    create_final_recommendation_report(
        low_positions[0]
    )
)

middle_report = (
    create_final_recommendation_report(
        middle_positions[0]
    )
)

high_report = (
    create_final_recommendation_report(
        high_positions[0]
    )
)

In [22]:
low_report

{'Student Position': 197,
 'Original Row Index': 197,
 'Predicted Employability Score': 2.229,
 'Strengths': [],
 'Development Recommendations': [{'Construct': 'Clear_Objectives',
   'Construct Score': 1.75,
   'SHAP Contribution': -0.6847,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'CO_R1',
     'text': 'Discuss internship goals and expected outcomes with the supervisor.'},
    {'id': 'CO_R2',
     'text': 'Break assigned tasks into clear weekly objectives.'},
    {'id': 'CO_R3',
     'text': 'Request clarification when tasks or performance expectations are unclear.'}]},
  {'Construct': 'School_Support',
   'Construct Score': 1.5,
   'SHAP Contribution': -0.517,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'SS_R1',
     'text': 'Schedule regular discussions with the academic internship supervisor.'},
    {'id': 'SS_R2

In [23]:
middle_report

{'Student Position': 989,
 'Original Row Index': 989,
 'Predicted Employability Score': 4.052,
 'Strengths': [{'Construct': 'Workplace_Environment',
   'Construct Score': 4.25,
   'SHAP Contribution': 0.015535661950707436},
  {'Construct': 'Career_Development',
   'Construct Score': 4.0,
   'SHAP Contribution': 0.00978376530110836}],
 'Development Recommendations': [],
 'Maintenance Message': 'No priority development area was identified. Maintain your current strengths in Workplace Environment, Career Development and continue seeking opportunities for professional development.',
 'Full Construct Analysis': [{'Construct': 'School_Support',
   'Construct Score': 4.0,
   'SHAP Contribution': -0.057614609599113464,
   'Score Level': 'High',
   'SHAP Effect': 'Moderate Negative',
   'Priority': 'Monitor'},
  {'Construct': 'Clear_Objectives',
   'Construct Score': 4.0,
   'SHAP Contribution': -0.019855670630931854,
   'Score Level': 'High',
   'SHAP Effect': 'Small Negative',
   'Priority': 

In [24]:
high_report

{'Student Position': 62,
 'Original Row Index': 62,
 'Predicted Employability Score': 4.725,
 'Strengths': [{'Construct': 'Clear_Objectives',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.21326914429664612},
  {'Construct': 'Workplace_Environment',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.17747583985328674},
  {'Construct': 'School_Support',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.1515226513147354},
  {'Construct': 'Career_Development',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.07898096740245819}],
 'Development Recommendations': [],
 'Maintenance Message': 'No priority development area was identified. Maintain your current strengths in Clear Objectives, Workplace Environment, School Support, Career Development and continue seeking opportunities for professional development.',
 'Full Construct Analysis': [{'Construct': 'Career_Development',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.07898096740245819,
   'Score Level': 'High',
   'SH

Generate reports for all records

In [25]:
all_final_reports = [
    create_final_recommendation_report(
        i
    )
    for i in range(
        len(X)
    )
]

print(
    "Reports generated:",
    len(
        all_final_reports
    )
)

Reports generated: 1000


Create summary dataframe

In [26]:
summary_rows = []

for report in (
    all_final_reports
):

    recommendations = (
        report[
            "Development Recommendations"
        ]
    )

    strengths = (
        report[
            "Strengths"
        ]
    )

    summary_rows.append({

        "Student Position":
            report[
                "Student Position"
            ],

        "Predicted Employability Score":
            report[
                "Predicted Employability Score"
            ],

        "Actual Employability Score":
            report[
                "Actual Employability Score"
            ],

        "Strengths":
            ", ".join(
                [
                    item[
                        "Construct"
                    ]
                    for item in strengths
                ]
            ),

        "Priority Areas":
            ", ".join(
                [
                    item[
                        "Construct"
                    ]
                    for item in recommendations
                ]
            ),

        "Recommendation Count":
            len(
                recommendations
            ),

        "Maintenance Feedback":
            report[
                "Maintenance Message"
            ]
    })

final_recommendation_summary = (
    pd.DataFrame(
        summary_rows
    )
)

final_recommendation_summary.head(
    10
)

,Student Position,Predicted Employability Score,Actual Employability Score,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback
0,0,4.396,4.625,"School_Support, Clear_Objectives, Career_Devel...",,0,No priority development area was identified. M...
1,1,4.077,3.875,Clear_Objectives,Career_Development,1,NaN
2,2,4.103,3.875,"Clear_Objectives, Career_Development",School_Support,1,NaN
3,3,3.878,4.000,,"Career_Development, School_Support",2,NaN
4,4,4.343,4.250,"Clear_Objectives, School_Support, Career_Devel...",,0,No priority development area was identified. M...
5,5,4.450,4.125,"Clear_Objectives, School_Support, Career_Devel...",,0,No priority development area was identified. M...
6,6,4.304,4.500,"Clear_Objectives, Career_Development, Workplac...",,0,No priority development area was identified. M...
7,7,3.778,3.250,,"Clear_Objectives, Career_Development",2,NaN
8,8,4.196,4.375,"Clear_Objectives, Career_Development",,0,No priority development area was identified. M...
9,9,4.005,4.125,Career_Development,,0,No priority development area was identified. M...


Targeted recommendation coverage

In [27]:
total_students = len(
    final_recommendation_summary
)

students_with_recommendations = (
    final_recommendation_summary[
        "Recommendation Count"
    ]
    .gt(0)
    .sum()
)

students_without_recommendations = (
    total_students
    - students_with_recommendations
)

coverage_percentage = (
    students_with_recommendations
    / total_students
    * 100
)

print(
    "Total students:",
    total_students
)

print(
    "Students with targeted recommendations:",
    students_with_recommendations
)

print(
    "Students with maintenance feedback:",
    students_without_recommendations
)

print(
    "Targeted recommendation coverage:",
    round(
        coverage_percentage,
        2
    ),
    "%"
)

Total students: 1000
Students with targeted recommendations: 209
Students with maintenance feedback: 791
Targeted recommendation coverage: 20.9 %


Recommendation frequency by construct

In [28]:
construct_counts = {
    feature: 0
    for feature in feature_names
}

for report in (
    all_final_reports
):

    for item in report[
        "Development Recommendations"
    ]:

        construct_counts[
            item[
                "Construct"
            ]
        ] += 1


recommendation_frequency = (
    pd.DataFrame({

        "Construct":
            construct_counts.keys(),

        "Students Recommended":
            construct_counts.values()
    })

    .sort_values(
        "Students Recommended",
        ascending=False
    )
)

recommendation_frequency

,Construct,Students Recommended
0,Clear_Objectives,115
3,Career_Development,86
2,Workplace_Environment,59
1,School_Support,32


Priority distribution

In [29]:
priority_values = []

for report in (
    all_final_reports
):

    for item in report[
        "Development Recommendations"
    ]:

        priority_values.append(
            item[
                "Priority"
            ]
        )


priority_distribution = (
    pd.Series(
        priority_values
    )

    .value_counts()

    .rename_axis(
        "Priority"
    )

    .reset_index(
        name="Count"
    )
)

priority_distribution

,Priority,Count
0,Moderate Priority,237
1,High Priority,53
2,Priority,2


Check empty feedback cases

In [30]:
empty_feedback_cases = []

for report in (
    all_final_reports
):

    has_recommendation = (
        len(
            report[
                "Development Recommendations"
            ]
        )
        > 0
    )

    has_maintenance = bool(
        report[
            "Maintenance Message"
        ]
    )

    if (
        not has_recommendation
        and not has_maintenance
    ):

        empty_feedback_cases.append(
            report[
                "Student Position"
            ]
        )

empty_feedback_cases

[]

Low-prediction without advice

In [31]:
low_prediction_without_advice = (
    final_recommendation_summary[

        (
            final_recommendation_summary[
                "Predicted Employability Score"
            ]
            < 3.0
        )

        &

        (
            final_recommendation_summary[
                "Recommendation Count"
            ]
            == 0
        )
    ]
)

low_prediction_without_advice

,Student Position,Predicted Employability Score,Actual Employability Score,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback


Coverage by prediction group

In [32]:
def prediction_group(
    score
):

    if score < 3.0:
        return "Lower"

    elif score < 4.0:
        return "Middle"

    return "Higher"

In [33]:
final_recommendation_summary[
    "Prediction Group"
] = (
    final_recommendation_summary[
        "Predicted Employability Score"
    ]
    .apply(
        prediction_group
    )
)

In [34]:
coverage_by_group = (

    final_recommendation_summary

    .groupby(
        "Prediction Group"
    )

    .agg(

        Students=(
            "Student Position",
            "count"
        ),

        Students_With_Recommendations=(
            "Recommendation Count",
            lambda x:
                (x > 0).sum()
        ),

        Average_Recommendations=(
            "Recommendation Count",
            "mean"
        )
    )
)

coverage_by_group

,Students,Students_With_Recommendations,Average_Recommendations
Prediction Group,,,
Higher,871,80,0.096441
Lower,8,8,2.000000
Middle,121,121,1.586777


Save revalidation results

In [35]:
final_recommendation_summary.to_csv(
    OUTPUT_DIR
    / "Final_XGBoost_Recommendation_Summary.csv",
    index=False
)

recommendation_frequency.to_csv(
    OUTPUT_DIR
    / "Final_Recommendation_Frequency.csv",
    index=False
)

priority_distribution.to_csv(
    OUTPUT_DIR
    / "Final_Priority_Distribution.csv",
    index=False
)

coverage_by_group.to_csv(
    OUTPUT_DIR
    / "Final_Coverage_By_Group.csv"
)

print(
    "Final recommendation validation files saved."
)

Final recommendation validation files saved.


## Final Recommendation Revalidation

Following final model selection, the recommendation engine was revalidated using the frozen XGBoost prediction model and its corresponding frozen SHAP thresholds.

The previously developed hybrid decision logic was retained without modification. Construct scores represented the student's current internship-experience profile, while local XGBoost SHAP values represented the direction and relative magnitude of each construct's contribution to the model prediction.

Technical checks examined recommendation coverage, priority distribution, construct-specific recommendation frequency, lower-prediction cases without targeted advice, and cases with empty feedback.

This stage evaluates the internal technical consistency of the deployment recommendation engine rather than predictive generalization. Predictive generalization was evaluated separately through nested cross-validation.

The recommendation content remains preliminary pending formal domain-expert validation.

for the clarify purposes

In [36]:
coverage_percentage

np.float64(20.9)

In [37]:
recommendation_frequency

,Construct,Students Recommended
0,Clear_Objectives,115
3,Career_Development,86
2,Workplace_Environment,59
1,School_Support,32


In [38]:
priority_distribution

,Priority,Count
0,Moderate Priority,237
1,High Priority,53
2,Priority,2


In [39]:
coverage_by_group

,Students,Students_With_Recommendations,Average_Recommendations
Prediction Group,,,
Higher,871,80,0.096441
Lower,8,8,2.000000
Middle,121,121,1.586777


In [40]:
empty_feedback_cases

[]

In [41]:
low_prediction_without_advice

,Student Position,Predicted Employability Score,Actual Employability Score,Strengths,Priority Areas,Recommendation Count,Maintenance Feedback


In [42]:
low_report

{'Student Position': 197,
 'Original Row Index': 197,
 'Predicted Employability Score': 2.229,
 'Strengths': [],
 'Development Recommendations': [{'Construct': 'Clear_Objectives',
   'Construct Score': 1.75,
   'SHAP Contribution': -0.6847,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'CO_R1',
     'text': 'Discuss internship goals and expected outcomes with the supervisor.'},
    {'id': 'CO_R2',
     'text': 'Break assigned tasks into clear weekly objectives.'},
    {'id': 'CO_R3',
     'text': 'Request clarification when tasks or performance expectations are unclear.'}]},
  {'Construct': 'School_Support',
   'Construct Score': 1.5,
   'SHAP Contribution': -0.517,
   'Priority': 'High Priority',
   'Reason': 'Low construct score with Large Negative model contribution',
   'Actions': [{'id': 'SS_R1',
     'text': 'Schedule regular discussions with the academic internship supervisor.'},
    {'id': 'SS_R2

In [43]:
high_report

{'Student Position': 62,
 'Original Row Index': 62,
 'Predicted Employability Score': 4.725,
 'Strengths': [{'Construct': 'Clear_Objectives',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.21326914429664612},
  {'Construct': 'Workplace_Environment',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.17747583985328674},
  {'Construct': 'School_Support',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.1515226513147354},
  {'Construct': 'Career_Development',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.07898096740245819}],
 'Development Recommendations': [],
 'Maintenance Message': 'No priority development area was identified. Maintain your current strengths in Clear Objectives, Workplace Environment, School Support, Career Development and continue seeking opportunities for professional development.',
 'Full Construct Analysis': [{'Construct': 'Career_Development',
   'Construct Score': 5.0,
   'SHAP Contribution': 0.07898096740245819,
   'Score Level': 'High',
   'SH

In [44]:
higher_with_recommendations = (
    final_recommendation_summary[
        (final_recommendation_summary["Prediction Group"] == "Higher")
        &
        (final_recommendation_summary["Recommendation Count"] > 0)
    ]
)

higher_with_recommendations[
    [
        "Student Position",
        "Predicted Employability Score",
        "Priority Areas",
        "Recommendation Count"
    ]
].head(10)

,Student Position,Predicted Employability Score,Priority Areas,Recommendation Count
1,1,4.077,Career_Development,1
2,2,4.103,School_Support,1
10,10,4.178,Career_Development,1
13,13,4.164,School_Support,1
19,19,4.080,Career_Development,1
20,20,4.164,School_Support,1
21,21,4.096,"Career_Development, School_Support",2
25,25,4.216,Career_Development,1
27,27,4.074,Career_Development,1
31,31,4.074,Career_Development,1


In [45]:
higher_position = int(
    higher_with_recommendations.iloc[0][
        "Student Position"
    ]
)

create_final_recommendation_report(
    higher_position
)

{'Student Position': 1,
 'Original Row Index': 1,
 'Predicted Employability Score': 4.077,
 'Strengths': [{'Construct': 'Clear_Objectives',
   'Construct Score': 4.25,
   'SHAP Contribution': 0.11998441815376282}],
 'Development Recommendations': [{'Construct': 'Career_Development',
   'Construct Score': 3.5,
   'SHAP Contribution': -0.0769,
   'Priority': 'Moderate Priority',
   'Reason': 'Moderate construct score with Large Negative model contribution',
   'Actions': [{'id': 'CD_R1',
     'text': 'Participate in challenging projects that build industry-relevant experience.'},
    {'id': 'CD_R2',
     'text': 'Seek networking, mentoring, and professional development opportunities.'},
    {'id': 'CD_R3',
     'text': 'Document achievements and skills gained during the internship.'}]}],
 'Maintenance Message': None,
 'Full Construct Analysis': [{'Construct': 'Career_Development',
   'Construct Score': 3.5,
   'SHAP Contribution': -0.07688244432210922,
   'Score Level': 'Moderate',
   'S